# 02. 프롬프트와 생성 파라미터 실험

같은 모델이라도 **프롬프트와 파라미터**에 따라 결과가 크게 달라집니다.
이 노트북에서는 생성 품질을 좌우하는 핵심 요소를 하나씩 바꿔가며 비교합니다.

1. `guidance_scale` — 프롬프트를 얼마나 충실히 따를지
2. `num_inference_steps` — 디노이징 단계 수 (품질 vs 속도)
3. 네거티브 프롬프트 — 원치 않는 요소 배제

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
print('준비 완료')

## 1. guidance_scale 비교

`guidance_scale`이 낮으면 자유롭게, 높으면 프롬프트를 강하게 따릅니다. 너무 높으면 부자연스러워질 수 있습니다.
같은 seed로 값만 바꿔 비교합니다.

In [ ]:
prompt = 'a majestic lion in a field of flowers, oil painting'
scales = [3.0, 7.5, 15.0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, scale in zip(axes, scales):
    g = torch.Generator(device=device).manual_seed(7)
    img = pipe(prompt, num_inference_steps=30, guidance_scale=scale, generator=g).images[0]
    ax.imshow(img); ax.axis('off'); ax.set_title(f'guidance_scale={scale}')
plt.tight_layout(); plt.show()

## 2. num_inference_steps 비교

디노이징 단계가 많을수록 품질이 좋아지지만 시간이 오래 걸립니다. 적정값(보통 25~50)을 찾는 것이 중요합니다.

In [ ]:
steps_list = [10, 25, 50]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, steps in zip(axes, steps_list):
    g = torch.Generator(device=device).manual_seed(7)
    img = pipe(prompt, num_inference_steps=steps, guidance_scale=7.5, generator=g).images[0]
    ax.imshow(img); ax.axis('off'); ax.set_title(f'steps={steps}')
plt.tight_layout(); plt.show()

## 3. 네거티브 프롬프트

`negative_prompt`로 원치 않는 특징(흐릿함, 왜곡 등)을 배제할 수 있습니다.

In [ ]:
g1 = torch.Generator(device=device).manual_seed(123)
img_no = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g1).images[0]

g2 = torch.Generator(device=device).manual_seed(123)
img_neg = pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
               negative_prompt='blurry, low quality, distorted', generator=g2).images[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(img_no); axes[0].axis('off'); axes[0].set_title('네거티브 없음')
axes[1].imshow(img_neg); axes[1].axis('off'); axes[1].set_title('네거티브 적용')
plt.tight_layout(); plt.show()

### 정리

- `guidance_scale`, `num_inference_steps`, 네거티브 프롬프트가 생성 결과에 미치는 영향을 비교했습니다.
- 같은 seed를 고정해 파라미터 효과만 분리해 관찰했습니다.
- 좋은 결과는 프롬프트 설계와 파라미터 튜닝의 조합에서 나옵니다.